In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini", stream_usage=True,temperature=0.7)


In [ ]:
from pydantic import BaseModel,Field
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict


class llm_schema(BaseModel):
    funny_flag:List['funny','not funny']=Field(description="Whether the joke is funny or not funny")
    feedback:str=Field(description="Feedback on the joke")



llm_with_schema=llm.with_structured_output(llm_schema)



SyntaxError: invalid syntax (1276378784.py, line 9)

In [ ]:
class graph_schema(TypedDict):

    topic:str
    joke:str
    funny_flag:str
    feedback:str
    max_iteration:int

In [ ]:
def generate_node(state:graph_schema)->graph_schema:

    topic=state['topic']
    
    if state['feedback']:
        response=llm.invoke(f"please modify thefollowing joke {joke} based on the following feedback {state['feedback']}")
    else:
        response=llm.invoke(f'create a joke about teh following topic {topic}')

    state['joke']=response.content
    return state



In [ ]:
def evaluate_node(state:graph_schema)->graph_schema:
    joke=state['joke']

    iteration=state[max_iteration]

    prompt=ChatPromptTemplate.from_messages([
        ("system":"you are a comedy critic youur job is to evaluate the following joke and provide feedback on how to make it funnier")
        ("user":f"evaluate the following joke {joke} and respond with funny or not funny and if it is not funny provide feedback about how to make it funny")
    ])

    chain=prompt|llm_with_schema
    response=chain.invoke({'joke':joke})
    state['funny_flag']=response.funny_flag
    state['feedback']=response.feedback
    state['max_iteration']=iteration+1

    return state

In [ ]:
def check_iteration(state:graph_schema)->graph_schema:

    iteration=state['max_iteration']

    if iteration <=5:
        return 'evaluate_node'
    else:
        return 'end'

In [ ]:
rom langgraph.graph import StateGraph, START, END

graph=StateGraph(graph_Schema)

graph.add_node("generate_node",generate_node)
graph.add_node("evaluate_node",evaluate_node)


graph.add_edge(START,'generate_node')
graph.add_conditional_edges('generate_node',check_iteration{'evaluate_node':'evaluate_node','end':END})
graph.add_edge("evaluate_node","generate_node")
graph.add_edge("generate_node",END)

In [ ]:

complex_graph=graph.compile()
Image(complex_graph.get_graph().draw_mermaid_png())